In [1]:
from pathlib import Path
import csv, json, shutil, zipfile

In [ ]:
root = Path(r"C:/Users/<USER>/<FOLDER>/dax-udf-patterns")
if root.exists(): shutil.rmtree(root)
for d in ['functions/period-to-date','functions/moving','functions/previous-period','functions/growth','functions/dynamic','distribution','docs']:
    (root/d).mkdir(parents=True, exist_ok=True)

In [3]:
items=[]
def add(folder, code, function, name, purpose, params, body, output='value'):
    docs='\n'.join([f'/// @param {{{typ}}} {n} - {desc}' for n,typ,desc in params])
    sig=',\n    '.join([f'{n} : {typ}' for n,typ,_ in params])
    text=f'''/// {purpose}
{docs}
/// @returns {"A decimal growth ratio." if output=="percentage" else "The calculated value for the requested time period."}
FUNCTION {function} = (
    {sig}
) =>
{body.strip()}
'''
    p=root/'functions'/folder/f'{code.lower()}.dax'; p.write_text(text,encoding='utf-8')
    items.append(dict(code=code,function=function,name=name,purpose=purpose,output=output,path=str(p.relative_to(root)).replace('\\','/')))

In [4]:
base=[('CalcValue','ANYREF EXPR','Measure or expression evaluated inside the function.'),('dateTable','ANYREF EXPR','Date table whose filters are cleared when explicit boundaries are applied.'),('dateColumn','ANYREF EXPR','Continuous date column from the marked date table.')]

In [5]:
def bounded(start,end):
 return f'''VAR _Result =
  CALCULATE(
    CalcValue,
    ALL(dateTable),
    dateColumn >= {start} && dateColumn <= {end}
  )
RETURN
  _Result'''

In [6]:
add('period-to-date','YTD','YearToDate','Year-to-date','Evaluates an expression from the first day of the current year through the latest visible date.',base,'''VAR _LatestDate = MAX(dateColumn)
VAR _StartDate = DATE(YEAR(_LatestDate), 1, 1)
'''+bounded('_StartDate','_LatestDate'))
add('period-to-date','QTD','QuarterToDate','Quarter-to-date','Evaluates an expression from the first day of the current quarter through the latest visible date.',base,'''VAR _LatestDate = MAX(dateColumn)
VAR _StartMonth = INT((MONTH(_LatestDate) - 1) / 3) * 3 + 1
VAR _StartDate = DATE(YEAR(_LatestDate), _StartMonth, 1)
'''+bounded('_StartDate','_LatestDate'))
add('period-to-date','MTD','MonthToDate','Month-to-date','Evaluates an expression from the first day of the current month through the latest visible date.',base,'''VAR _LatestDate = MAX(dateColumn)
VAR _StartDate = DATE(YEAR(_LatestDate), MONTH(_LatestDate), 1)
'''+bounded('_StartDate','_LatestDate'))
add('moving','MAT','MovingAnnualTotal','Moving annual total','Evaluates an expression for the 12-month window ending on the latest visible date.',base,'''VAR _EndDate = MAX(dateColumn)
VAR _StartDate = EDATE(_EndDate, -12) + 1
'''+bounded('_StartDate','_EndDate'))

In [7]:
for code,fn,name,unit in [('PY','PriorYear','Previous year','YEAR'),('PQ','PriorQuarter','Previous quarter','QUARTER'),('PM','PriorMonth','Previous month','MONTH')]:
 add('previous-period',code,fn,name,f'Evaluates an expression in the current date context shifted back one {unit.lower()}.',base,f'''VAR _Result =
  CALCULATE(
    CalcValue,
    DATEADD(dateColumn, -1, {unit})
  )
RETURN
  _Result''')

In [8]:
add('previous-period','PYC','PriorYearComplete','Previous year complete','Evaluates an expression for the complete calendar year before the current year.',base,'''VAR _LatestDate = MAX(dateColumn)
VAR _StartDate = DATE(YEAR(_LatestDate) - 1, 1, 1)
VAR _EndDate = DATE(YEAR(_LatestDate) - 1, 12, 31)
'''+bounded('_StartDate','_EndDate'))
add('previous-period','PQC','PriorQuarterComplete','Previous quarter complete','Evaluates an expression for the complete quarter before the current quarter.',base,'''VAR _LatestDate = MAX(dateColumn)
VAR _CurrentStartMonth = INT((MONTH(_LatestDate) - 1) / 3) * 3 + 1
VAR _CurrentStart = DATE(YEAR(_LatestDate), _CurrentStartMonth, 1)
VAR _StartDate = EDATE(_CurrentStart, -3)
VAR _EndDate = _CurrentStart - 1
'''+bounded('_StartDate','_EndDate'))
add('previous-period','PMC','PriorMonthComplete','Previous month complete','Evaluates an expression for the complete month before the current month.',base,'''VAR _LatestDate = MAX(dateColumn)
VAR _EndDate = EOMONTH(_LatestDate, -1)
VAR _StartDate = DATE(YEAR(_EndDate), MONTH(_EndDate), 1)
'''+bounded('_StartDate','_EndDate'))

In [9]:
scope=base+[('yearColumn','ANYREF EXPR','Year hierarchy column.'),('quarterColumn','ANYREF EXPR','Quarter hierarchy column.'),('monthColumn','ANYREF EXPR','Month hierarchy column.')]
add('dynamic','PP','PriorPeriod','Previous period','Selects the previous month, quarter, or year according to hierarchy scope.',scope,'''VAR _Result =
  SWITCH(
    TRUE(),
    ISINSCOPE(monthColumn), CALCULATE(CalcValue, DATEADD(dateColumn, -1, MONTH)),
    ISINSCOPE(quarterColumn), CALCULATE(CalcValue, DATEADD(dateColumn, -1, QUARTER)),
    ISINSCOPE(yearColumn), CALCULATE(CalcValue, DATEADD(dateColumn, -1, YEAR)),
    BLANK()
  )
RETURN
  _Result''')

In [10]:
add('moving','PYMAT','PriorYearMovingAnnualTotal','Previous year moving annual total','Evaluates the comparable 12-month window ending one year before the latest visible date.',base,'''VAR _CurrentEnd = MAX(dateColumn)
VAR _EndDate = EDATE(_CurrentEnd, -12)
VAR _StartDate = EDATE(_EndDate, -12) + 1
'''+bounded('_StartDate','_EndDate'))

In [11]:
def growth_body(unit):
 return f'''VAR _CurrentValue = CalcValue
VAR _PriorValue = CALCULATE(CalcValue, DATEADD(dateColumn, -1, {unit}))
VAR _Result = DIVIDE(_CurrentValue - _PriorValue, _PriorValue)
RETURN
  _Result'''
for code,fn,name,unit in [('YOY','YearOverYear','Year-over-year','YEAR'),('QOQ','QuarterOverQuarter','Quarter-over-quarter','QUARTER'),('MOM','MonthOverMonth','Month-over-month','MONTH')]:
 add('growth',code,fn,name,f'Returns growth versus the comparable date context one {unit.lower()} earlier.',base,growth_body(unit),'percentage')

In [12]:
add('growth','MATG','MovingAnnualTotalGrowth','Moving annual total growth','Returns growth between the current moving annual total and the comparable prior-year window.',base,'''VAR _CurrentEnd = MAX(dateColumn)
VAR _CurrentStart = EDATE(_CurrentEnd, -12) + 1
VAR _PriorEnd = EDATE(_CurrentEnd, -12)
VAR _PriorStart = EDATE(_PriorEnd, -12) + 1
VAR _CurrentValue = CALCULATE(CalcValue, ALL(dateTable), dateColumn >= _CurrentStart && dateColumn <= _CurrentEnd)
VAR _PriorValue = CALCULATE(CalcValue, ALL(dateTable), dateColumn >= _PriorStart && dateColumn <= _PriorEnd)
VAR _Result = DIVIDE(_CurrentValue - _PriorValue, _PriorValue)
RETURN
  _Result''','percentage')
add('dynamic','POP','PeriodOverPeriod','Period-over-period','Returns growth versus the previous month, quarter, or year according to hierarchy scope.',scope,'''VAR _CurrentValue = CalcValue
VAR _PriorValue =
  SWITCH(
    TRUE(),
    ISINSCOPE(monthColumn), CALCULATE(CalcValue, DATEADD(dateColumn, -1, MONTH)),
    ISINSCOPE(quarterColumn), CALCULATE(CalcValue, DATEADD(dateColumn, -1, QUARTER)),
    ISINSCOPE(yearColumn), CALCULATE(CalcValue, DATEADD(dateColumn, -1, YEAR)),
    BLANK()
  )
VAR _Result = DIVIDE(_CurrentValue - _PriorValue, _PriorValue)
RETURN
  _Result''','percentage')

In [13]:
# Explicit elapsed-period functions. Cap corresponding end dates at prior period end.
prior_bodies={
'PYTD':('PriorYearToDate','Previous year-to-date','''VAR _LatestDate = MAX(dateColumn)
VAR _StartDate = DATE(YEAR(_LatestDate) - 1, 1, 1)
VAR _CandidateEnd = DATE(YEAR(_LatestDate) - 1, MONTH(_LatestDate), DAY(_LatestDate))
VAR _EndDate = MIN(_CandidateEnd, DATE(YEAR(_LatestDate) - 1, 12, 31))
'''),
'PQTD':('PriorQuarterToDate','Previous quarter-to-date','''VAR _LatestDate = MAX(dateColumn)
VAR _CurrentStartMonth = INT((MONTH(_LatestDate) - 1) / 3) * 3 + 1
VAR _CurrentStart = DATE(YEAR(_LatestDate), _CurrentStartMonth, 1)
VAR _ElapsedDays = _LatestDate - _CurrentStart
VAR _StartDate = EDATE(_CurrentStart, -3)
VAR _PeriodEnd = _CurrentStart - 1
VAR _EndDate = MIN(_StartDate + _ElapsedDays, _PeriodEnd)
'''),
'PMTD':('PriorMonthToDate','Previous month-to-date','''VAR _LatestDate = MAX(dateColumn)
VAR _CurrentStart = DATE(YEAR(_LatestDate), MONTH(_LatestDate), 1)
VAR _ElapsedDays = _LatestDate - _CurrentStart
VAR _StartDate = EDATE(_CurrentStart, -1)
VAR _PeriodEnd = _CurrentStart - 1
VAR _EndDate = MIN(_StartDate + _ElapsedDays, _PeriodEnd)
''')}
for code,(fn,name,prefix) in prior_bodies.items():
 add('period-to-date',code,fn,name,f'Evaluates an expression through the corresponding elapsed point in the prior {name.split()[1].replace("-to-date","")}.',base,prefix+bounded('_StartDate','_EndDate'))

In [14]:
# Growth PTD functions
for code,fn,name,currentfn,priorfn in [
 ('YOYTD','YearOverYearToDate','Year-over-year-to-date','YearToDate','PriorYearToDate'),
 ('QOQTD','QuarterOverQuarterToDate','Quarter-over-quarter-to-date','QuarterToDate','PriorQuarterToDate'),
 ('MOMTD','MonthOverMonthToDate','Month-over-month-to-date','MonthToDate','PriorMonthToDate')]:
 add('growth',code,fn,name,f'Returns growth between current period-to-date and the corresponding prior period-to-date.',base,f'''VAR _CurrentValue = {currentfn}(CalcValue, dateTable, dateColumn)
VAR _PriorValue = {priorfn}(CalcValue, dateTable, dateColumn)
VAR _Result = DIVIDE(_CurrentValue - _PriorValue, _PriorValue)
RETURN
  _Result''','percentage')

In [15]:
for code,fn,name,currentfn,priorfn in [
 ('YTDOPY','YearToDateOverPriorYear','Year-to-date over previous year','YearToDate','PriorYearComplete'),
 ('QTDOPQ','QuarterToDateOverPriorQuarter','Quarter-to-date over previous quarter','QuarterToDate','PriorQuarterComplete'),
 ('MTDOPM','MonthToDateOverPriorMonth','Month-to-date over previous month','MonthToDate','PriorMonthComplete')]:
 add('growth',code,fn,name,'Returns growth between the current period-to-date and the complete immediately preceding period.',base,f'''VAR _CurrentValue = {currentfn}(CalcValue, dateTable, dateColumn)
VAR _PriorValue = {priorfn}(CalcValue, dateTable, dateColumn)
VAR _Result = DIVIDE(_CurrentValue - _PriorValue, _PriorValue)
RETURN
  _Result''','percentage')

In [16]:
# Combined DAX definition file ordered so called functions come first.
order=['YTD','QTD','MTD','MAT','PY','PQ','PM','PYC','PQC','PMC','PP','PYMAT','YOY','QOQ','MOM','MATG','POP','PYTD','PQTD','PMTD','YOYTD','QOQTD','MOMTD','YTDOPY','QTDOPQ','MTDOPM']
by={x['code']:x for x in items}
combined='DEFINE\n\n'+'\n\n'.join((root/by[c]['path']).read_text(encoding='utf-8') for c in order)+'\n'
(root/'distribution'/'all-functions.dax').write_text(combined,encoding='utf-8')
# PBIP model-level TMDL distribution, transform FUNCTION to function and omit DEFINE.
tmdl='// Add these model-level functions to definition/functions.tmdl in a PBIP project.\n\n'
for c in order:
    s=(root/by[c]['path']).read_text(encoding='utf-8').replace('FUNCTION ','function ',1)
    tmdl+=s+'\n'
(root/'distribution'/'functions.tmdl').write_text(tmdl,encoding='utf-8')

21906

In [17]:
with (root/'manifest.csv').open('w',newline='',encoding='utf-8') as f:
 w=csv.DictWriter(f,fieldnames=['code','function','name','purpose','output','path']);w.writeheader();w.writerows(items)
(root/'manifest.json').write_text(json.dumps(items,indent=2),encoding='utf-8')
(root/'VERSION').write_text('0.1.0\n',encoding='utf-8')
(root/'.gitignore').write_text('*.tmp\n*.bak\n.DS_Store\nThumbs.db\n.vscode/\n',encoding='utf-8')
(root/'README.md').write_text('''# DAX Time Intelligence UDF Library

A Git-ready collection of reusable DAX user-defined functions. The functions accept the calculation expression, date table, and date column as arguments, so no physical table or measure names are embedded in the library.

## Distributions

- `distribution/all-functions.dax`: Query-view `DEFINE FUNCTION` definitions for testing and model updates.
- `distribution/functions.tmdl`: Model-level function definitions for a PBIP `definition/functions.tmdl` file.
- `functions/`: One reviewable `.dax` file per function.

## Standard call

```DAX
CPT Code Distribution Value YTD =
  YearToDate(
    [CPT Code Distribution Value],
    'dimDates',
    'dimDates'[DateValue]
  )
```

Dynamic functions also receive hierarchy columns:

```DAX
CPT Code Distribution Value PP =
  PriorPeriod(
    [CPT Code Distribution Value],
    'dimDates',
    'dimDates'[DateValue],
    'dimDates'[Year],
    'dimDates'[Quarter],
    'dimDates'[Month]
  )
```

## Assumptions

- Calendar-year logic is used for complete year and quarter patterns.
- MAT is an inclusive trailing 12-month period ending on the latest visible date.
- Growth functions return `(current - prior) / prior` through `DIVIDE`.
- PP and POP return blank when year, quarter, or month is not in hierarchy scope.
- The date table contains a continuous date column and is marked as a date table.

## Git workflow

1. Edit a single function file.
2. Update the matching entry in both distribution files.
3. Run the checks in `docs/testing.md`.
4. Update `CHANGELOG.md` and `VERSION` for released behavior changes.
5. Submit a pull request with expected and actual test results.
''',encoding='utf-8')
(root/'CONTRIBUTING.md').write_text('''# Contributing

- Preserve public function names unless making a documented breaking change.
- Keep model object names out of function bodies.
- Use underscore-prefixed DAX variables in proper case.
- Use `DIVIDE` for ratios.
- Keep one function per source file.
- Update both distribution files and the manifest when a function changes.
- Document calendar versus fiscal behavior explicitly.
''',encoding='utf-8')
(root/'CHANGELOG.md').write_text('''# Changelog

## 0.1.0

- Added 26 reusable DAX time-intelligence UDFs.
- Added combined DAX Query View and PBIP TMDL distributions.
- Added manifests, usage examples, and testing guidance.
''',encoding='utf-8')
(root/'docs'/'testing.md').write_text('''# Testing Checklist

- Confirm the semantic model compatibility level supports DAX UDFs.
- Confirm the date table has one row per date and no gaps.
- Test completed and partial year, quarter, and month contexts.
- Test month-end, quarter-end, year-end, and leap-day behavior.
- Test zero and blank prior values for all growth functions.
- Test PP and POP at year, quarter, month, and card scope.
- Compare each UDF result with a manually verified measure.
- Confirm totals behave as documented.
''',encoding='utf-8')
(root/'docs'/'naming.md').write_text('''# Public Function Names

YearToDate, QuarterToDate, MonthToDate, MovingAnnualTotal, PriorYear, PriorQuarter, PriorMonth, PriorYearComplete, PriorQuarterComplete, PriorMonthComplete, PriorPeriod, PriorYearMovingAnnualTotal, YearOverYear, QuarterOverQuarter, MonthOverMonth, MovingAnnualTotalGrowth, PeriodOverPeriod, PriorYearToDate, PriorQuarterToDate, PriorMonthToDate, YearOverYearToDate, QuarterOverQuarterToDate, MonthOverMonthToDate, YearToDateOverPriorYear, QuarterToDateOverPriorQuarter, MonthToDateOverPriorMonth.
''',encoding='utf-8')

522

In [ ]:
zip_path=Path(r"C:/Users/<USER>/<FOLDER>/dax-udf-patterns.zip")
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for p in root.rglob('*'):
  if p.is_file(): z.write(p,Path(root.name)/p.relative_to(root))
print(len(items), 'functions')
print(zip_path)